In [ ]:
import pandas as pd
import numpy as np
import re
import os
import matplotlib.pyplot as plt
import tqdm
from sklearn.decomposition import PCA

In [ ]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path("/Users/aniluchavez/Documents/Language/Python/Final_words_english_only_BERT")
# BASE_DIR = Path("/Users/aniluchavez/Documents/Language/Python/final_word2vecW")
all_data = []
#*_words_word2vec or *_words_english_only
for patient_folder in BASE_DIR.glob("*_words_english_only"):
    if patient_folder.is_dir():
        excel_files = [f for f in patient_folder.glob("*filtered_used_rows_withNP.xlsx") if not f.name.startswith("~$")]
        # excel_files = [f for f in patient_folder.glob("*filtered_used_rows_word2vec.xlsx") if not f.name.startswith("~$")]
        if not excel_files:
            print(f"⚠️  No Excel file found in {patient_folder}")
            continue

        excel_path = excel_files[0]
        print(f"✅ Reading: {excel_path.name}")

        try:
            df = pd.read_excel(excel_path)

            if 'Duration' not in df.columns:
                print(f"❌ Duration column missing in {excel_path}")
                continue

            # Find speaker columns
            speaker_cols = [col for col in df.columns if "Speaker" in col]
            if not speaker_cols:
                print(f"❌ No speaker columns found in {excel_path}. Columns were: {df.columns}")
                continue

            # Melt speaker columns to long
            melted = df.melt(
                id_vars=['Duration'],
                value_vars=speaker_cols,
                var_name='speaker',
                value_name='word'
            )

            # Drop empty
            melted = melted.dropna(subset=['word'])

            # Add patient_id
            patient_id = patient_folder.name.replace("_words_english_only", "")
            # patient_id = patient_folder.name.replace("_words_word2vec", "")
            melted['patient_id'] = patient_id

            # Keep columns
            final_cols = melted[['patient_id', 'word', 'Duration']]
            all_data.append(final_cols)

        except Exception as e:
            print(f"❌ Error reading {excel_path}: {e}")

# Combine
if all_data:
    combined_df = pd.concat(all_data, ignore_index=True)
    print(combined_df.head())

    output_file = BASE_DIR / "all_patients_words_durations.csv"
    combined_df.to_csv(output_file, index=False)
    print(f"✅ Combined CSV saved at: {output_file}")
else:
    print("❌ No data was combined.")


In [ ]:
df=all_data
import pandas as pd
import numpy as np
from pathlib import Path

def to_dataframe(maybe_df):
    """Coerce several common structures into a single pandas DataFrame."""
    # Already a DataFrame
    if isinstance(maybe_df, pd.DataFrame):
        return maybe_df

    # List cases
    if isinstance(maybe_df, list):
        if len(maybe_df) == 0:
            raise ValueError("Input is an empty list.")
        # list of DataFrames
        if all(isinstance(x, pd.DataFrame) for x in maybe_df):
            return pd.concat(maybe_df, ignore_index=True)
        # list of dict-like (records)
        if all(isinstance(x, dict) for x in maybe_df):
            return pd.DataFrame(maybe_df)
        # mixed / other - try a best-effort conversion
        try:
            return pd.DataFrame(maybe_df)
        except Exception as e:
            raise TypeError(f"Can't convert list to DataFrame automatically: {e}")

    # Dict-like objects
    if isinstance(maybe_df, dict):
        try:
            # If values are lists of same length this becomes columns
            return pd.DataFrame(maybe_df)
        except Exception:
            # Fallback: wrap dict as single-row record
            return pd.DataFrame([maybe_df])

    raise TypeError(f"Unsupported input type: {type(maybe_df)}. Expected DataFrame, list, or dict.")

# -------------------------
# Replace `all_data` with whatever variable you actually have
# e.g. df = all_data
# -------------------------
try:
    df = to_dataframe(all_data)   # <-- change variable name if different
except Exception as e:
    print("ERROR converting input to DataFrame:", e)
    # show quick introspection if available
    try:
        print("Type(all_data) =", type(all_data))
        if isinstance(all_data, list) and len(all_data) > 0:
            print("Sample element type:", type(all_data[0]))
            # print a small sample safely
            import itertools, json
            sample = all_data[:3]
            print("Sample elements (up to 3):")
            for i,s in enumerate(sample):
                print(f" [{i}] ->", type(s))
                # print small repr for dicts / scalars
                if isinstance(s, dict):
                    keys = list(s.keys())[:10]
                    print("      keys:", keys)
                else:
                    print("      repr:", repr(s)[:200])
    except Exception:
        pass
    raise

# quick debug print
print("Converted to DataFrame: shape =", df.shape)
print("Columns:", list(df.columns[:50]))
print("Head:")
print(df.head(3).to_string(index=False))

# try to locate the Duration column (case-insensitive search)
duration_cols = [c for c in df.columns if 'dur' in c.lower()]
if not duration_cols:
    raise KeyError("No column with 'dur' found in DataFrame columns. Columns: " + ", ".join(df.columns))

# pick the best candidate
duration_col = duration_cols[0]
print(f"Using duration column: '{duration_col}'")

# coerce to numeric ms and drop rows lacking durations
df[duration_col] = pd.to_numeric(df[duration_col], errors='coerce')
n_missing = df[duration_col].isna().sum()
if n_missing:
    print(f"Warning: {n_missing} rows have non-numeric or missing durations and will be dropped.")
df = df.dropna(subset=[duration_col]).reset_index(drop=True)

# compute WPM (robust)
total_words = len(df)
total_duration_ms = df[duration_col].sum()
total_duration_min = total_duration_ms / 60000.0
wpm_overall = total_words / total_duration_min if total_duration_min > 0 else np.nan
mean_word_dur = df[duration_col].mean()
wpm_from_mean = 60000.0 / mean_word_dur if mean_word_dur > 0 else np.nan

print("\n=== Overall WPM ===")
print(f"Total words (rows): {total_words:,}")
print(f"Total duration (min): {total_duration_min:.2f}")
print(f"WPM (by sum of durations): {wpm_overall:.2f}")
print(f"WPM (by mean duration): {wpm_from_mean:.2f}")
print(f"Mean word duration (ms): {mean_word_dur:.1f}")
print(f"Median word duration (ms): {df[duration_col].median():.1f}")

# Per-patient breakdown if patient column exists (try common names)
patient_cols = [c for c in df.columns if c.lower() in ('patient', 'patient_id', 'patientid')]
if patient_cols:
    pcol = patient_cols[0]
    per_patient = (
        df.groupby(pcol)[duration_col]
        .agg(n_words='count', total_dur_ms='sum', mean_dur_ms='mean', median_dur_ms='median')
        .reset_index()
    )
    per_patient['total_dur_min'] = per_patient['total_dur_ms'] / 60000.0
    per_patient['wpm_by_sum'] = per_patient['n_words'] / per_patient['total_dur_min']
    per_patient['wpm_by_mean'] = 60000.0 / per_patient['mean_dur_ms']
    print(f"\nPer-patient WPM summary using patient column '{pcol}':")
    print(per_patient.sort_values('wpm_by_sum', ascending=False).head(10).to_string(index=False))
    # save CSV next to notebook
    outp = Path.cwd() / "per_patient_wpm_summary.csv"
    per_patient.to_csv(outp, index=False)
    print(f"\nSaved per-patient summary to: {outp}")
else:
    print("\nNo obvious patient column found (checked: patient, patient_id, patientid). Skipping per-patient breakdown.")



In [ ]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path("/Users/aniluchavez/Documents/Language/Python/final_word2vecW")

# === Desired patient order (prefix match) ===
patient_order = ["PTYFA", "PTYEU", "PTYFK", "PTYFC", "PTYEY","PTYEZ","PTYFG", "PTYFF", "PTYFI", "PTYEV" ]

# === Find all matching folders and sort them by the custom order ===
all_folders = list(BASE_DIR.glob("*_words_word2vec"))
ordered_folders = []

for prefix in patient_order:
    for folder in all_folders:
        if folder.name.startswith(prefix):
            ordered_folders.append(folder)
            break
    else:
        print(f"⚠️ Folder not found for prefix: {prefix}")

# === Process each folder in order ===
all_data = []

for patient_folder in ordered_folders:
    if patient_folder.is_dir():
        excel_files = [f for f in patient_folder.glob("*filtered_used_rows_word2vec.xlsx") if not f.name.startswith("~$")]
        if not excel_files:
            print(f"⚠️  No Excel file found in {patient_folder}")
            continue

        excel_path = excel_files[0]
        print(f"✅ Reading: {excel_path.name}")

        try:
            df = pd.read_excel(excel_path)

            if 'Duration' not in df.columns:
                print(f"❌ Duration column missing in {excel_path}")
                continue

            speaker_cols = [col for col in df.columns if "Speaker" in col]
            if not speaker_cols:
                print(f"❌ No speaker columns found in {excel_path}. Columns were: {df.columns}")
                continue

            melted = df.melt(
                id_vars=['Duration'],
                value_vars=speaker_cols,
                var_name='speaker',
                value_name='word'
            )

            melted = melted.dropna(subset=['word'])

            patient_id = patient_folder.name.replace("_words_word2vec", "")
            melted['patient_id'] = patient_id

            final_cols = melted[['patient_id', 'word', 'Duration']]
            all_data.append(final_cols)

        except Exception as e:
            print(f"❌ Error reading {excel_path}: {e}")

# === Combine and save ===
if all_data:
    combined_df = pd.concat(all_data, ignore_index=True)
    print(combined_df.head())

    output_file = BASE_DIR / "all_patients_words_durations.csv"
    combined_df.to_csv(output_file, index=False)
    print(f"✅ Combined CSV saved at: {output_file}")
else:
    print("❌ No data was combined.")


# Pie

In [ ]:
import math
import re
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

# === CONFIG ===
BASE_DIRS = [
    Path("/Users/aniluchavez/Documents/Language/Python/Final_words_english_only_BERT"),
    Path("/Users/aniluchavez/Documents/Language/Python/final_word2vecW")
]

OUT_DIR = Path("/Users/aniluchavez/Documents/Language/Python/pie_charts_per_patient")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Toggle: only final-word folders, or include word2vec folders as well
include_word2vec = False

# Show mode: "percent" -> show percent, "count" -> show raw number, "both" -> show both
display_mode = "count"   # choose: "percent", "count", or "both"

# Main plotting options
target_speaker = "Speaker1"
colors = ["#EF5193", "#607D38"]  # Self, Other
save_eps = True
save_png = True

# Combined subplot config
page_size = 30      # max pies per page
ncols = 6           # columns per page
title_fontsize = 9
autopct_fontsize = 7


# === Helper functions ===
def find_patient_folders(base_dir: Path, include_word2vec=False):
    folders = list(base_dir.glob("*_words_english_only"))
    if include_word2vec:
        folders += list(base_dir.glob("*_words_word2vec"))
    return sorted(set(folders), key=lambda p: p.name)


def autopct_factory(sizes, mode="percent"):
    total = sum(sizes) if sizes is not None else None

    def _autopct_percent(pct):
        return f"{pct:.1f}%"

    def _autopct_count(pct):
        if total is None:
            return f"{int(round(pct))}"
        val = int(round(pct * total / 100.0))
        return f"{val}"

    def _autopct_both(pct):
        if total is None:
            return f"{pct:.1f}%"
        val = int(round(pct * total / 100.0))
        return f"{pct:.1f}%\n({val})"

    if mode == "percent":
        return _autopct_percent
    elif mode == "count":
        return _autopct_count
    else:
        return _autopct_both


def is_target_speaker_col(col, target_speaker):
    """
    Robust check whether 'col' refers to target_speaker.
    1) If both contain digits, compare digits (so Speaker1 matches SPK1 but not Speaker10).
    2) Otherwise compare normalized alphanumeric strings (removes spaces/punct).
    """
    col_s = str(col).lower().strip()
    tgt_s = str(target_speaker).lower().strip()

    mcol = re.search(r'(\d+)', col_s)
    mtgt = re.search(r'(\d+)', tgt_s)
    if mcol and mtgt:
        return mcol.group(1) == mtgt.group(1)

    def normalize(x):
        return re.sub(r'[\W_]+', '', x)  # remove non-alphanumeric and underscores

    return normalize(col_s) == normalize(tgt_s)


def find_preferred_excel(pf: Path):
    """
    Prefer filtered_used_rows_withNP variants, else fallback to any .xlsx (not temp files).
    Returns Path or None.
    """
    matches = [f for f in pf.glob("*filtered_used_rows_withNP*.xlsx") if not f.name.startswith("~$")]
    if matches:
        return sorted(matches, key=lambda p: p.name)[0]
    matches = [f for f in pf.glob("*.xlsx") if not f.name.startswith("~$")]
    if matches:
        return sorted(matches, key=lambda p: p.name)[0]
    return None


# === Collect per-patient stats ===
patient_stats = []

for base in BASE_DIRS:
    if not base.exists():
        print(f"⚠️ Base dir not found: {base}")
        continue

    patient_folders = find_patient_folders(base, include_word2vec)
    if not patient_folders:
        print(f"⚠️ No patient folders found under {base} (include_word2vec={include_word2vec})")
        continue

    for pf in patient_folders:
        excel_path = find_preferred_excel(pf)
        if excel_path is None:
            print(f"⚠️ No excel found in {pf}")
            continue

        try:
            df = pd.read_excel(excel_path)
        except Exception as e:
            print(f"❌ Error reading {excel_path}: {e}")
            continue

        # detect columns that look like speaker columns
        speaker_cols = [c for c in df.columns if "speaker" in str(c).lower()]
        if not speaker_cols:
            print(f"❌ No speaker columns in {excel_path}. Columns: {list(df.columns)}")
            continue

        id_vars = ["Duration"] if "Duration" in df.columns else []
        melted = df.melt(id_vars=id_vars, value_vars=speaker_cols,
                         var_name="speaker_col", value_name="word_token")
        melted = melted.dropna(subset=["word_token"])
        melted["speaker_col"] = melted["speaker_col"].astype(str)

        # assign roles with robust matcher
        melted["role"] = melted["speaker_col"].apply(
            lambda s: "self" if is_target_speaker_col(s, target_speaker) else "other"
        )

        # debug: show which columns are treated as self for this patient
        self_cols = sorted({c for c in melted['speaker_col'].unique() if is_target_speaker_col(c, target_speaker)})
        if self_cols:
            print(f"Patient {pf.name}: Columns treated as SELF for target '{target_speaker}': {self_cols}")
        else:
            print(f"Patient {pf.name}: No columns matched target speaker '{target_speaker}' (check naming).")

        counts = melted["role"].value_counts()
        count_self = int(counts.get("self", 0))
        count_other = int(counts.get("other", 0))

        total = count_self + count_other
        if total == 0:
            print(f"⚠️ No tokens for patient folder {pf.name}")
            continue

        # === per-patient pie (individual file) ===
        sizes = [count_self, count_other]
        autopct_fn = autopct_factory(sizes, mode=display_mode)

        fig, ax = plt.subplots(figsize=(5, 5))
        wedges, texts, autotexts = ax.pie(
            sizes,
            labels=["Self", "Other"],
            autopct=autopct_fn,
            startangle=90,
            colors=colors,
            explode=(0.05, 0.0),
            wedgeprops={"linewidth": 0.6, "edgecolor": "white"},
            textprops={"fontsize": 10}
        )
        ax.axis("equal")
        ax.set_title(f"{pf.name} — tokens: self={count_self}, other={count_other}", fontsize=10)
        filename_base = OUT_DIR / f"{pf.name}_self_vs_other_pie"
        try:
            if save_eps:
                plt.savefig(str(filename_base) + ".eps", format="eps", bbox_inches="tight")
            if save_png:
                plt.savefig(str(filename_base) + ".png", dpi=150, bbox_inches="tight")
            print(f"✅ Saved pie for {pf.name} -> self:{count_self} other:{count_other}")
        except Exception as e:
            print(f"❌ Error saving pie for {pf.name}: {e}")
        finally:
            plt.close(fig)

        patient_stats.append({
            "name": pf.name,
            "self": count_self,
            "other": count_other,
            "total": total
        })

# === Combined subplot pages ===
if not patient_stats:
    print("❌ No patient stats collected; nothing to plot in combined figure.")
else:
    patient_stats = sorted(patient_stats, key=lambda x: x["name"])
    n_patients = len(patient_stats)
    pages = math.ceil(n_patients / page_size)
    print(f"Creating {pages} combined page(s) for {n_patients} patients (page_size={page_size}).")

    for page_idx in range(pages):
        start = page_idx * page_size
        end = min(start + page_size, n_patients)
        page_stats = patient_stats[start:end]
        n_page = len(page_stats)
        nrows = math.ceil(n_page / ncols)

        fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(ncols*2.2, nrows*2.2))
        # normalize axes to a flat list for easy indexing
        if isinstance(axes, (list, tuple)):
            axes_flat = [ax for row in axes for ax in (row if hasattr(row, "__iter__") else [row])]
        else:
            axes_flat = list(axes.flatten()) if hasattr(axes, "flatten") else [axes]

        # turn off unused axes
        for ax in axes_flat[n_page:]:
            try:
                ax.axis("off")
            except Exception:
                pass

        for ax, ps in zip(axes_flat, page_stats):
            sizes = [ps["self"], ps["other"]]
            autopct_fn = autopct_factory(sizes, mode=display_mode)
            wedges = ax.pie(
                sizes,
                labels=None,
                autopct=autopct_fn,
                startangle=90,
                colors=colors,
                wedgeprops={"linewidth": 0.4, "edgecolor": "white"},
                textprops={"fontsize": autopct_fontsize}
            )
            ax.axis("equal")
            ax.set_title(f"{ps['name']}\nS:{ps['self']} O:{ps['other']}", fontsize=title_fontsize)

        # global legend (one per page)
        fig.legend(["Self", "Other"], loc="lower center", ncol=2, fontsize=9)
        fig.suptitle(f"Patient token split (page {page_idx+1}/{pages}) — display: {display_mode}", fontsize=11)
        plt.tight_layout(rect=[0, 0.04, 1, 0.96])

        page_filename_base = OUT_DIR / f"all_patients_pies_page{page_idx+1}_display-{display_mode}"
        try:
            if save_eps:
                plt.savefig(str(page_filename_base) + ".eps", format="eps", bbox_inches="tight")
            if save_png:
                plt.savefig(str(page_filename_base) + ".png", dpi=150, bbox_inches="tight")
            print(f"✅ Saved combined page {page_idx+1} with {n_page} pies -> {page_filename_base}.png/.eps")
        except Exception as e:
            print(f"❌ Error saving combined page {page_idx+1}: {e}")
        finally:
            plt.close(fig)

print("All done.")


In [ ]:
import re
import math
import pandas as pd
from pathlib import Path
from collections import Counter
import json

# === CONFIG (edit to match your environment) ===
BASE_DIRS = [
    Path("/Users/aniluchavez/Documents/Language/Python/Final_words_english_only_BERT"),
    # add other base dirs if you want
]
include_word2vec = False   # keep consistent with your earlier run
target_speaker = "Speaker1"
OUT_DIR = Path("/Users/aniluchavez/Documents/Language/Python/pie_charts_per_patient")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Toggle token normalization
normalize_tokens = True

# --- helper functions (same logic as your pie script) ---
def is_target_speaker_col(col, target_speaker):
    col_s = str(col).lower().strip()
    tgt_s = str(target_speaker).lower().strip()
    mcol = re.search(r'(\d+)', col_s)
    mtgt = re.search(r'(\d+)', tgt_s)
    if mcol and mtgt:
        return mcol.group(1) == mtgt.group(1)
    def normalize(x):
        return re.sub(r'[\W_]+', '', x)  # remove non-alphanumeric and underscores
    return normalize(col_s) == normalize(tgt_s)

def find_patient_folders(base_dir):
    folders = list(base_dir.glob("*_words_english_only"))
    # optionally include word2vec variant
    return sorted(set(folders), key=lambda p: p.name)

def find_preferred_excel(pf: Path):
    matches = [f for f in pf.glob("*filtered_used_rows_withNP*.xlsx") if not f.name.startswith("~$")]
    if matches:
        return sorted(matches, key=lambda p: p.name)[0]
    matches = [f for f in pf.glob("*.xlsx") if not f.name.startswith("~$")]
    if matches:
        return sorted(matches, key=lambda p: p.name)[0]
    return None

# small normalizer for tokens (keeps internal apostrophes, lowercases)
_token_re = re.compile(r"^[^\w']+|[^\w']+$")  # trim non-word/apostrophe at ends
def normalize_token(tok):
    if not isinstance(tok, str):
        return tok
    t = tok.strip().lower()
    t = _token_re.sub("", t)
    return t

# === collect metrics per patient ===
rows = []
overall_self_vocab = set()
overall_other_vocab = set()

# NEW: shared-token trackers
overall_shared_vocab = set()            # set of tokens that were shared in at least one patient
per_patient_shared_counts = []          # list of dicts for per-patient shared metrics
global_shared_counter = Counter()       # counts of shared-token occurrences aggregated across patients

for base in BASE_DIRS:
    if not base.exists():
        print(f"⚠️ Base dir not found: {base}")
        continue

    patient_folders = find_patient_folders(base)
    if not patient_folders:
        continue

    for pf in patient_folders:
        excel = find_preferred_excel(pf)
        if excel is None:
            #print(f"No excel for {pf.name}")
            continue

        try:
            df = pd.read_excel(excel)
        except Exception as e:
            print(f"Error reading {excel}: {e}")
            continue

        speaker_cols = [c for c in df.columns if "speaker" in str(c).lower()]
        if not speaker_cols:
            #print(f"No speaker columns in {excel}")
            continue

        id_vars = ["Duration"] if "Duration" in df.columns else []
        melted = df.melt(id_vars=id_vars, value_vars=speaker_cols,
                         var_name="speaker_col", value_name="word_token")
        melted = melted.dropna(subset=["word_token"]).reset_index(drop=True)
        # classify role
        melted['role'] = melted['speaker_col'].apply(lambda s: "self" if is_target_speaker_col(s, target_speaker) else "other")

        # optionally normalize tokens
        if normalize_tokens:
            melted['token_norm'] = melted['word_token'].astype(str).apply(normalize_token)
            # drop purely-empty tokens after normalization
            melted = melted[melted['token_norm'].notna() & (melted['token_norm'] != '')].copy()
            token_col = 'token_norm'
        else:
            token_col = 'word_token'

        # compute counts
        self_tokens = melted[melted['role'] == 'self'][token_col].tolist()
        other_tokens = melted[melted['role'] == 'other'][token_col].tolist()

        n_self = len(self_tokens)
        n_other = len(other_tokens)
        unique_self = set(self_tokens)
        unique_other = set(other_tokens)

        # update overall vocabs
        overall_self_vocab.update(unique_self)
        overall_other_vocab.update(unique_other)

        # per-patient unique counts
        nuniq_self = len(unique_self)
        nuniq_other = len(unique_other)

        # type-token ratio (unique / total) safe-guard
        ttr_self = nuniq_self / n_self if n_self > 0 else float('nan')
        ttr_other = nuniq_other / n_other if n_other > 0 else float('nan')

        # NEW: shared-word metrics for this patient
        shared_tokens = unique_self.intersection(unique_other)   # set of tokens present in both roles
        n_shared_unique = len(shared_tokens)

        # occurrences among shared tokens (counts within self + other for tokens in the intersection)
        # Build per-role counters
        self_counter = Counter(self_tokens)
        other_counter = Counter(other_tokens)

        # For this patient, sum occurrences of shared tokens across both roles
        shared_occurrences_self = sum(self_counter[t] for t in shared_tokens)
        shared_occurrences_other = sum(other_counter[t] for t in shared_tokens)
        shared_occurrences_total = shared_occurrences_self + shared_occurrences_other

        # update global trackers
        overall_shared_vocab.update(shared_tokens)
        # add to global_shared_counter token-wise (we count combined occurrences here)
        for t in shared_tokens:
            global_shared_counter[t] += self_counter.get(t, 0) + other_counter.get(t, 0)

        per_patient_shared_counts.append({
            "patient": pf.name,
            "n_shared_unique": n_shared_unique,
            "shared_occ_self": int(shared_occurrences_self),
            "shared_occ_other": int(shared_occurrences_other),
            "shared_occ_total": int(shared_occurrences_total)
        })

        rows.append({
            "patient": pf.name,
            "n_self_tokens": n_self,
            "n_other_tokens": n_other,
            "nuniq_self_tokens": nuniq_self,
            "nuniq_other_tokens": nuniq_other,
            "ttr_self": ttr_self,
            "ttr_other": ttr_other,
            "n_shared_unique": n_shared_unique,
            "shared_occ_total": int(shared_occ_total := shared_occurrences_total)
        })

# Make DF(s)
metrics_df = pd.DataFrame(rows).sort_values("patient").reset_index(drop=True)
per_patient_shared_df = pd.DataFrame(per_patient_shared_counts).sort_values("patient").reset_index(drop=True)

out_csv = OUT_DIR / "per_patient_word_unique_metrics_with_shared.csv"
metrics_df.to_csv(out_csv, index=False)
per_patient_shared_df.to_csv(OUT_DIR / "per_patient_shared_word_metrics.csv", index=False)
print(f"Saved per-patient metrics to: {out_csv}")
print(f"Saved per-patient shared metrics to: {OUT_DIR/'per_patient_shared_word_metrics.csv'}")

# === Aggregate / averages across patients ===
self_nonzero = metrics_df[metrics_df['n_self_tokens'] > 0]
other_nonzero = metrics_df[metrics_df['n_other_tokens'] > 0]

avg_self_tokens_per_patient = self_nonzero['n_self_tokens'].mean() if len(self_nonzero) else float('nan')
avg_other_tokens_per_patient = other_nonzero['n_other_tokens'].mean() if len(other_nonzero) else float('nan')

avg_self_unique_per_patient = self_nonzero['nuniq_self_tokens'].mean() if len(self_nonzero) else float('nan')
avg_other_unique_per_patient = other_nonzero['nuniq_other_tokens'].mean() if len(other_nonzero) else float('nan')

median_self_tokens_per_patient = self_nonzero['n_self_tokens'].median() if len(self_nonzero) else float('nan')
median_other_tokens_per_patient = other_nonzero['n_other_tokens'].median() if len(other_nonzero) else float('nan')

# shared summary
shared_nonzero = per_patient_shared_df[per_patient_shared_df['n_shared_unique'] > 0]
avg_shared_unique_per_patient = shared_nonzero['n_shared_unique'].mean() if len(shared_nonzero) else 0.0
median_shared_unique_per_patient = shared_nonzero['n_shared_unique'].median() if len(shared_nonzero) else 0.0
avg_shared_occ_total_per_patient = shared_nonzero['shared_occ_total'].mean() if len(shared_nonzero) else 0.0
median_shared_occ_total_per_patient = shared_nonzero['shared_occ_total'].median() if len(shared_nonzero) else 0.0

# overall vocab sizes across all patients
overall_nuniq_self = len(overall_self_vocab)
overall_nuniq_other = len(overall_other_vocab)
overall_n_shared_tokens = len(overall_shared_vocab)

# totals
total_self_tokens = metrics_df['n_self_tokens'].sum()
total_other_tokens = metrics_df['n_other_tokens'].sum()

print("\n=== Summary across patients ===")
print(f"Patients with self tokens: {len(self_nonzero)}")
print(f"Patients with other tokens: {len(other_nonzero)}")
print(f"Average words per patient (Speaker1/self): {avg_self_tokens_per_patient:.2f}")
print(f"Average words per patient (Others):           {avg_other_tokens_per_patient:.2f}")
print(f"Median words per patient (Speaker1/self):  {median_self_tokens_per_patient:.1f}")
print(f"Median words per patient (Others):          {median_other_tokens_per_patient:.1f}")
print()
print(f"Average unique words per patient (Speaker1/self): {avg_self_unique_per_patient:.2f}")
print(f"Average unique words per patient (Others):         {avg_other_unique_per_patient:.2f}")
print()
print(f"Average shared UNIQUE words per patient (only patients with any shared words): {avg_shared_unique_per_patient:.2f}")
print(f"Median shared UNIQUE words per patient: {median_shared_unique_per_patient:.1f}")
print(f"Average shared-occurrences total per patient (self+other): {avg_shared_occ_total_per_patient:.2f}")
print(f"Median shared-occurrences total per patient: {median_shared_occ_total_per_patient:.1f}")
print()
print(f"Overall unique vocabulary across all patients (Speaker1/self): {overall_nuniq_self}")
print(f"Overall unique vocabulary across all patients (Others)       : {overall_nuniq_other}")
print(f"Overall number of distinct tokens that were SHARED in >=1 patient: {overall_n_shared_tokens}")
print()
print(f"Total tokens summed across patients (Speaker1/self): {total_self_tokens}")
print(f"Total tokens summed across patients (Others)         : {total_other_tokens}")

# === Top-10 shared tokens across patients ===
top_shared = global_shared_counter.most_common(50)  # top 50 to give context
top10_shared = top_shared[:10]
top10_df = pd.DataFrame(top10_shared, columns=['token', 'shared_total_occurrences'])
top10_df.to_csv(OUT_DIR / "top10_shared_tokens_across_patients.csv", index=False)

print("\nTop 10 shared tokens (token, total shared occurrences across patients):")
print(top10_df.to_string(index=False))

# Save overall summary JSON
summary = {
    "n_patients": len(metrics_df),
    "avg_self_tokens_per_patient": avg_self_tokens_per_patient,
    "avg_other_tokens_per_patient": avg_other_tokens_per_patient,
    "avg_self_unique_per_patient": avg_self_unique_per_patient,
    "avg_other_unique_per_patient": avg_other_unique_per_patient,
    "avg_shared_unique_per_patient": avg_shared_unique_per_patient,
    "median_shared_unique_per_patient": median_shared_unique_per_patient,
    "avg_shared_occ_total_per_patient": avg_shared_occ_total_per_patient,
    "median_shared_occ_total_per_patient": median_shared_occ_total_per_patient,
    "overall_nuniq_self": overall_nuniq_self,
    "overall_nuniq_other": overall_nuniq_other,
    "overall_n_shared_tokens": overall_n_shared_tokens,
    "total_self_tokens": int(total_self_tokens),
    "total_other_tokens": int(total_other_tokens)
}
with open(OUT_DIR / "word_vocab_summary_with_shared.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"\nSaved top-10 shared tokens CSV to: {OUT_DIR/'top10_shared_tokens_across_patients.csv'}")
print(f"Saved overall summary JSON to: {OUT_DIR/'word_vocab_summary_with_shared.json'}")

if 'metrics_df' not in globals() or metrics_df is None or metrics_df.shape[0] == 0:
    raise RuntimeError("metrics_df not found or empty — run the prior processing block first.")

# compute per-patient fractions (shared unique / speaker unique)
df = metrics_df.copy()

# Avoid division-by-zero
df['frac_shared_of_self_unique'] = df.apply(
    lambda r: (r['n_shared_unique'] / r['nuniq_self_tokens']) if r['nuniq_self_tokens'] and not np.isnan(r['nuniq_self_tokens']) else np.nan,
    axis=1
)
df['frac_shared_of_other_unique'] = df.apply(
    lambda r: (r['n_shared_unique'] / r['nuniq_other_tokens']) if r['nuniq_other_tokens'] and not np.isnan(r['nuniq_other_tokens']) else np.nan,
    axis=1
)

# Stats across patients (ignore patients where denominator was zero -> NaN)
self_frac_nonan = df['frac_shared_of_self_unique'].dropna()
other_frac_nonan = df['frac_shared_of_other_unique'].dropna()

summary = {
    'per_patient_frac_shared_of_self_unique_mean': float(self_frac_nonan.mean()) if len(self_frac_nonan) else None,
    'per_patient_frac_shared_of_self_unique_median': float(self_frac_nonan.median()) if len(self_frac_nonan) else None,
    'per_patient_frac_shared_of_other_unique_mean': float(other_frac_nonan.mean()) if len(other_frac_nonan) else None,
    'per_patient_frac_shared_of_other_unique_median': float(other_frac_nonan.median()) if len(other_frac_nonan) else None,
    'avg_shared_unique_per_patient': float(df['n_shared_unique'].dropna().mean()),
    'median_shared_unique_per_patient': float(df['n_shared_unique'].dropna().median()),
    'avg_shared_occ_total_per_patient': float(df['shared_occ_total'].dropna().mean()) if 'shared_occ_total' in df.columns else None
}

# Overall union-based fraction (across-patients vocab)
overall_nuniq_self = len(overall_self_vocab) if 'overall_self_vocab' in globals() else None
overall_n_shared_tokens = len(overall_shared_vocab) if 'overall_shared_vocab' in globals() else None
if overall_nuniq_self and overall_n_shared_tokens is not None:
    summary['overall_shared_types_overall_self_types_frac'] = overall_n_shared_tokens / overall_nuniq_self
else:
    summary['overall_shared_types_overall_self_types_frac'] = None

# add readable percentages
if summary['per_patient_frac_shared_of_self_unique_mean'] is not None:
    summary['per_patient_frac_shared_of_self_unique_mean_pct'] = summary['per_patient_frac_shared_of_self_unique_mean'] * 100.0
if summary['per_patient_frac_shared_of_self_unique_median'] is not None:
    summary['per_patient_frac_shared_of_self_unique_median_pct'] = summary['per_patient_frac_shared_of_self_unique_median'] * 100.0
if summary['overall_shared_types_overall_self_types_frac'] is not None:
    summary['overall_shared_types_overall_self_types_frac_pct'] = summary['overall_shared_types_overall_self_types_frac'] * 100.0

# print friendly summary
print("\n===== SHARED / UNIQUE FRACTIONS SUMMARY =====")
print(f"Mean fraction of Speaker1's UNIQUE tokens that are shared (per-patient average): "
      f"{summary['per_patient_frac_shared_of_self_unique_mean']:.3f} "
      f"({summary['per_patient_frac_shared_of_self_unique_mean_pct']:.1f}%)")
print(f"Median fraction (per-patient): {summary['per_patient_frac_shared_of_self_unique_median']:.3f} "
      f"({summary['per_patient_frac_shared_of_self_unique_median_pct']:.1f}%)")
print()
if summary['overall_shared_types_overall_self_types_frac'] is not None:
    print(f"Across-patients: {overall_n_shared_tokens} distinct tokens were SHARED somewhere, out of "
          f"{overall_nuniq_self} distinct Speaker1 token types in the whole dataset -> "
          f"{summary['overall_shared_types_overall_self_types_frac']:.3f} "
          f"({summary['overall_shared_types_overall_self_types_frac_pct']:.1f}%)")
print()
print(f"Average shared unique tokens per patient (repeated): {summary['avg_shared_unique_per_patient']:.2f}")
print(f"Average shared-occurrences total per patient (self+other): {summary['avg_shared_occ_total_per_patient']:.2f}")
print("==============================================\n")

# Save per-patient fraction table and overall summary JSON
df.to_csv(OUT_DIR / "per_patient_shared_fraction_table.csv", index=False)
with open(OUT_DIR / "shared_fraction_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"Saved per-patient fraction table to: {OUT_DIR/'per_patient_shared_fraction_table.csv'}")
print(f"Saved overall shared-fraction summary JSON to: {OUT_DIR/'shared_fraction_summary.json'}")

# Optional: produce the top-10 shared tokens by (a) number of patients in which they were shared,
# and (b) by total shared occurrences (we computed global_shared_counter earlier).
if 'global_shared_counter' in globals():
    # token -> number of patients it was shared in — rebuild if you have token->patientset mapping
    # if you ran the token->patientset version previously you can reuse it. Here we approximate by just
    # using counts from global_shared_counter (already aggregated shared occurrences).
    top_by_occ = global_shared_counter.most_common(20)
    top10_by_occ = pd.DataFrame(top_by_occ[:10], columns=['token','shared_total_occurrences'])
    top10_by_occ.to_csv(OUT_DIR / "top10_shared_tokens_by_occurrences.csv", index=False)
    print("Top-10 shared tokens by total shared occurrences (saved):")
    print(top10_by_occ.to_string(index=False))
else:
    print("global_shared_counter not found — top-10 by occurrences not generated.")


In [ ]:
import math
import re
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd
from pathlib import Path
from textwrap import wrap

# === CONFIG === (edit paths if needed)
BASE_DIRS = [
    Path("/Users/aniluchavez/Documents/Language/Python/Final_words_english_only_BERT"),
    Path("/Users/aniluchavez/Documents/Language/Python/final_word2vecW")
]

OUT_DIR = Path("/Users/aniluchavez/Documents/Language/Python/pie_charts_per_patient")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Save outputs
summary_csv = OUT_DIR / "all_patients_speaker_counts.csv"     # long form
pivot_csv = OUT_DIR / "all_patients_speaker_pivot.csv"        # wide form (patients x speakers)

# Plotting options
save_eps = True
save_png = True
display_mode = "count"   # kept for compatibility
autopct_fontsize = 8
figsize_per_patient = (6, 6)

# Title wrapping options
title_chunk_size = 6      # number of "speaker=count" pairs per title line for individual pies
grid_title_chunk_size = 4 # for combined multi-patient grid (smaller)

# === Helpers ===
def find_patient_folders(base_dir: Path, include_word2vec=False):
    folders = list(base_dir.glob("*_words_english_only"))
    if include_word2vec:
        folders += list(base_dir.glob("*_words_word2vec"))
    return sorted(set(folders), key=lambda p: p.name)

def find_preferred_excel(pf: Path):
    matches = [f for f in pf.glob("*filtered_used_rows_withNP.xlsx") if not f.name.startswith("~$")]
    if matches:
        return sorted(matches, key=lambda p: p.name)[0]
    matches = [f for f in pf.glob("*.xlsx") if not f.name.startswith("~$")]
    if matches:
        return sorted(matches, key=lambda p: p.name)[0]
    return None

def format_speaker_counts_for_title(items, chunk_size=6):
    pairs = [f"{k}={v}" for k, v in items]
    lines = [", ".join(pairs[i:i+chunk_size]) for i in range(0, len(pairs), chunk_size)]
    return "\n".join(lines)

def extract_first_digits(s):
    m = re.search(r'(\d+)', str(s))
    return m.group(1) if m else None

def canonical_speaker_key(colname):
    """
    Normalize a speaker column into a canonical key:
      - If column name contains digits -> 'speaker{N}' (lowercase)
      - Else -> normalized column string (lower, no spaces)
    This key is used for consistent color mapping across files.
    """
    col = str(colname).strip()
    d = extract_first_digits(col)
    if d:
        return f"speaker{int(d)}"
    # fallback normalize text
    return re.sub(r'[\W_]+', '_', col.lower()).strip('_')

def make_color_map(speaker_keys):
    """
    Given an ordered list of speaker keys, return a dict key->hexcolor.
    Uses matplotlib's tab20 (or generates N colors) for as many keys as needed.
    """
    n = len(speaker_keys)
    if n <= 20:
        cmap = plt.get_cmap("tab20")
        colors = [mcolors.to_hex(cmap(i)) for i in range(n)]
    else:
        # create N distinct colors via hsv sampling
        colors = []
        for i in range(n):
            hue = i / n
            rgb = mcolors.hsv_to_rgb((hue, 0.6, 0.9))
            colors.append(mcolors.to_hex(rgb))
    return dict(zip(speaker_keys, colors))

# === PASS 1: scan all speaker columns across all patient files to build global speaker set ===
all_speaker_keys = []
seen = set()

for base in BASE_DIRS:
    if not base.exists():
        print(f"⚠️ Base dir not found: {base}")
        continue

    patient_folders = find_patient_folders(base, include_word2vec=False)
    for pf in patient_folders:
        excel_path = find_preferred_excel(pf)
        if excel_path is None:
            continue
        try:
            df = pd.read_excel(excel_path, nrows=0)  # only need columns
        except Exception:
            continue
        # detect speaker-like columns
        speaker_cols = [c for c in df.columns if re.search(r'(speaker|spk)\s*\d+', str(c), flags=re.I)]
        if not speaker_cols:
            speaker_cols = [c for c in df.columns if ('speaker' in str(c).lower() and re.search(r'\d+', str(c)))]
        for c in speaker_cols:
            key = canonical_speaker_key(c)
            if key not in seen:
                seen.add(key)
                all_speaker_keys.append(key)

if not all_speaker_keys:
    print("⚠️ No speaker columns found in any scanned files. Exiting.")
    raise SystemExit(1)

# Build color map keyed by canonical speaker key
color_map = make_color_map(all_speaker_keys)

# Convenience: when creating colors for pie labels, convert label -> canonical key -> color
def colors_for_labels(labels):
    cols = []
    for lab in labels:
        key = canonical_speaker_key(lab)
        # if key unknown (unlikely), fall back to a default color map entry or gray
        cols.append(color_map.get(key, "#888888"))
    return cols

# === MAIN: collect counts and produce pies ===
rows = []
for base in BASE_DIRS:
    if not base.exists():
        print(f"⚠️ Base dir not found: {base}")
        continue

    patient_folders = find_patient_folders(base, include_word2vec=False)
    if not patient_folders:
        print(f"⚠️ No patient folders under {base}")
        continue

    for pf in patient_folders:
        excel_path = find_preferred_excel(pf)
        if excel_path is None:
            print(f"⚠️ No excel found in {pf}")
            continue

        try:
            df = pd.read_excel(excel_path)
        except Exception as e:
            print(f"❌ Error reading {excel_path}: {e}")
            continue

        # detect speaker-like columns that include a numeric suffix (avoid stray '...' cols)
        speaker_cols = [c for c in df.columns if re.search(r'(speaker|spk)\s*\d+', str(c), flags=re.I)]
        if not speaker_cols:
            speaker_cols = [c for c in df.columns if ('speaker' in str(c).lower() and re.search(r'\d+', str(c)))]
        if not speaker_cols:
            print(f"❌ No speaker columns found in {excel_path} (expected Speaker## style). Skipping.")
            continue

        # compute raw non-null counts per speaker column
        speaker_counts = {str(c): int(df[c].notna().sum()) for c in speaker_cols}

        # save rows for CSV
        for spk_col, cnt in speaker_counts.items():
            rows.append({
                "patient_folder": pf.name,
                "excel_path": str(excel_path),
                "speaker_col": spk_col,
                "count": cnt
            })

        # prepare pie inputs (sorted descending)
        items = sorted(speaker_counts.items(), key=lambda kv: kv[1], reverse=True)
        labels = [k for k, v in items]
        sizes = [v for k, v in items]

        if sum(sizes) == 0:
            print(f"⚠️ Patient {pf.name} has zero tokens across detected speaker columns; skipping pie.")
            continue

        # compute colors for these labels using canonical mapping
        slice_colors = colors_for_labels(labels)

        # plot (no numbers inside slices)
        fig, ax = plt.subplots(figsize=figsize_per_patient)
        ax.pie(sizes, labels=labels, colors=slice_colors, autopct=None, startangle=90,
               wedgeprops={"linewidth": 0.6, "edgecolor": "white"}, textprops={"fontsize": autopct_fontsize})
        ax.axis("equal")

        # title: patient name + wrapped speaker=count lines
        speaker_title = format_speaker_counts_for_title(items, chunk_size=title_chunk_size)
        ax.set_title(f"{pf.name}\n{speaker_title}", fontsize=9)

        out_base = OUT_DIR / f"{pf.name}_per_speaker_pie"
        try:
            if save_eps:
                plt.savefig(str(out_base) + ".eps", format="eps", bbox_inches="tight")
            if save_png:
                plt.savefig(str(out_base) + ".png", dpi=150, bbox_inches="tight")
            print(f"✅ Saved pie for {pf.name} -> {out_base}.eps/.png")
        except Exception as e:
            print(f"❌ Error saving pie for {pf.name}: {e}")
        finally:
            plt.close(fig)

# Save summary CSVs (long and pivot)
df_long = pd.DataFrame(rows)
if not df_long.empty:
    df_long.to_csv(summary_csv, index=False)
    pivot = df_long.pivot_table(index="patient_folder", columns="speaker_col", values="count", aggfunc="sum").fillna(0).astype(int)
    pivot.to_csv(pivot_csv)
    print(f"✅ Saved long CSV: {summary_csv}")
    print(f"✅ Saved pivot CSV: {pivot_csv}")

    # === Combined multi-patient grid with speaker=count in subplot titles ===
    patients = df_long["patient_folder"].unique()
    n_patients = len(patients)
    if n_patients > 0:
        n_cols = math.ceil(math.sqrt(n_patients))
        n_rows = math.ceil(n_patients / n_cols)
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*5, n_rows*5))
        axes_flat = axes.flatten() if hasattr(axes, "flatten") else [axes]

        for i, patient in enumerate(patients):
            subdf = df_long[df_long["patient_folder"] == patient]
            counts = subdf.groupby("speaker_col")["count"].sum().sort_values(ascending=False)
            labels = counts.index.tolist()
            sizes = counts.values.tolist()
            ax = axes_flat[i]
            slice_colors = colors_for_labels(labels)
            ax.pie(sizes, labels=labels, colors=slice_colors, autopct=None, startangle=90,
                   wedgeprops={"linewidth": 0.6, "edgecolor": "white"}, textprops={"fontsize": autopct_fontsize})
            ax.axis("equal")
            # format a shorter wrapped title for grid (smaller chunk size)
            speaker_title = format_speaker_counts_for_title(list(zip(counts.index, counts.values)), chunk_size=grid_title_chunk_size)
            ax.set_title(f"{patient}\n{speaker_title}", fontsize=8)

        # remove unused subplots
        for j in range(i+1, len(axes_flat)):
            fig.delaxes(axes_flat[j])

        plt.tight_layout()
        combined_path = OUT_DIR / "all_patients_pies"
        try:
            if save_eps:
                plt.savefig(str(combined_path) + ".eps", format="eps", bbox_inches="tight")
            if save_png:
                plt.savefig(str(combined_path) + ".png", dpi=150, bbox_inches="tight")
            print(f"✅ Saved combined multi-patient pies: {combined_path}.eps/.png")
        except Exception as e:
            print(f"❌ Error saving combined multi-patient pies: {e}")
        finally:
            plt.close(fig)
else:
    print("No speaker counts collected; nothing saved.")


# This assigns the cluster IDs to each indiv patient excel

In [ ]:
import pandas as pd

# Load data
word2vec_df = pd.read_csv("/Users/aniluchavez/Documents/Language/Python/final_word2vecW/all_patients_word2vecwords.csv")
clustered_df = pd.read_csv("/Users/aniluchavez/Documents/Language/Updated_Clustered_Words_Finale.csv")

# Save original order
word2vec_df["original_index"] = word2vec_df.index

# Create fast lookup from cluster list
cluster_lookup = dict(zip(clustered_df["text"], clustered_df["FinalClusterID"]))

# Assign cluster ID by word
word2vec_df["FinalClusterID"] = word2vec_df["word"].map(cluster_lookup)

# Restore original order
word2vec_df = word2vec_df.sort_values(by="original_index").drop(columns=["original_index"])

# Save final result
word2vec_df.to_csv("word2vec_clustered_order_restored.csv", index=False)
print("✅ Cluster IDs assigned and order restored. Saved to: word2vec_clustered_order_restored.csv")


# Word duration histogram 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Set global font to Arial
plt.rcParams['font.family'] = 'Arial'

# Load your CSV
csv_path = "/Users/aniluchavez/Documents/Language/Python/Final_words_english_only_BERT/all_patients_words_durations.csv"
df = pd.read_csv(csv_path)

# Ensure Duration is numeric
df['Duration'] = pd.to_numeric(df['Duration'], errors='coerce')
df = df.dropna(subset=['Duration'])

# Make the figure
fig, ax = plt.subplots(figsize=(10, 8))

# Plot the histogram
ax.hist(df['Duration'], bins=70, color='#64c6c2', edgecolor='black')
ax.set_xlim(0, 1200)

# Labels with big fonts
ax.set_xlabel('Duration (ms)', fontsize=30)
ax.set_ylabel('Count', fontsize=30)

# Tick labels big
ax.tick_params(axis='both', which='major', labelsize=24)

# Clean style
ax.grid(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(1.5)
ax.spines['bottom'].set_linewidth(1.5)

plt.tight_layout()

save_path = "/Users/aniluchavez/Documents/Language/Figures/Figure1/word_duration_histogram.eps"
plt.savefig(save_path, format='eps', dpi=300)

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Set global font
plt.rcParams['font.family'] = 'Arial'

# Load and clean data
csv_path = "/Users/aniluchavez/Documents/Language/Python/Final_words_english_only_BERT/all_patients_words_durations.csv"
df = pd.read_csv(csv_path)
df['Duration'] = pd.to_numeric(df['Duration'], errors='coerce')
df = df.dropna(subset=['Duration'])

# Limit to durations within plot x-axis
df = df[df['Duration'] <= 1500]

# === Sample 14 words evenly across duration range ===
quantiles = np.linspace(0.0, 1, 80)
dur_cutoffs = df['Duration'].quantile(quantiles).values
sample_words = []

for i in range(len(dur_cutoffs) - 1):
    sub_df = df[(df['Duration'] >= dur_cutoffs[i]) & (df['Duration'] < dur_cutoffs[i+1])]
    if not sub_df.empty:
        sample = sub_df.sample(1).squeeze()
        sample_words.append(sample)

# === Plot with wider bins ===
fig, ax = plt.subplots(figsize=(10, 8))

# Use wider bins (10 chunks over 0–1200)
counts, bins, _ = ax.hist(df['Duration'], bins=30, color='#64c6c2', edgecolor='black')
ax.set_xlim(0, 1200)

# Labels with big fonts
ax.set_xlabel('Duration (ms)', fontsize=20)
ax.set_ylabel('Count', fontsize=30)
ax.tick_params(axis='both', which='major', labelsize=24)

# Clean style
ax.grid(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(1.5)
ax.spines['bottom'].set_linewidth(1.5)

# === Add slanted word annotations ===
for word_info in sample_words:
    duration = word_info['Duration']
    word = word_info['word']

    bin_idx = np.digitize(duration, bins) - 1
    bin_idx = np.clip(bin_idx, 0, len(counts) - 1)
    y_pos = counts[bin_idx]

    ax.text(duration, y_pos + 500, word, rotation=45, ha='left', va='bottom', fontsize=14, color='black')

plt.tight_layout()

# Save
save_path = "/Users/aniluchavez/Documents/Language/Figures/Figure1/word_duration_histogram5_teset9.eps"
plt.savefig(save_path, format='eps', dpi=300)
plt.show()


# Word speaker turns

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Set global font
plt.rcParams['font.family'] = 'Arial'

# Path to your Excel file
EXCEL_PATH = "/Users/aniluchavez/Documents/Language/Python/Final_words_english_only_BERT/PTYEZ_task60_words_english_only/PTYEZ_task60_filtered_used_rows_withNP.xlsx"

# Load the Excel file
df = pd.read_excel(EXCEL_PATH)

# Inspect columns
print("Columns:", df.columns)

# Speaker columns in your file
speaker_columns = ["Speaker1", "Speaker2", "Speaker3"]

# Melt from wide to long format
long_df = df.melt(
    id_vars=["onset", "offset", "Duration", "regress_dur"],
    value_vars=speaker_columns,
    var_name="speaker",
    value_name="word"
)

# Drop rows without words
long_df = long_df.dropna(subset=["word"]).reset_index(drop=True)

# Sort by onset to preserve time order
long_df = long_df.sort_values(by="onset").reset_index(drop=True)

# Display a few rows to confirm
print("\n=== Example of long-format data ===")
print(long_df.head())

# Detect where the speaker changes
long_df["speaker_shift"] = long_df["speaker"] != long_df["speaker"].shift()

# Assign turn IDs
long_df["turn_id"] = long_df["speaker_shift"].cumsum()

# Create summary table of each turn
turn_summary = long_df.groupby("turn_id").agg({
    "speaker": "first",       # Which speaker
    "onset": "min",           # Start time of turn
    "word": "count"           # Number of words
}).rename(columns={"word": "n_words"}).reset_index()

# Show first 10 turns
print("\n=== Turn-level summary (first 10 turns) ===")
print(turn_summary.head(10))

# Show overall stats
print("\n=== Stats on words per turn ===")
print(turn_summary["n_words"].describe())

# Save full report to CSV
turn_summary.to_csv("turn_summary_report.csv", index=False)
print("\n✅ Full turn report saved to 'turn_summary_report.csv'")

# Plot histogram
plt.figure(figsize=(8, 5))
plt.hist(turn_summary["n_words"], bins=100, edgecolor='black')
plt.xlabel("Words per speaker turn")
plt.ylabel("Number of turns")
plt.title("Histogram of Words per Speaker Turn")
ax.grid(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(1.5)
ax.spines['bottom'].set_linewidth(1.5)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np

sorted_sizes = np.sort(turn_sizes)
cumulative = np.arange(1, len(sorted_sizes)+1) / len(sorted_sizes)

plt.figure(figsize=(8,5))
plt.plot(sorted_sizes, cumulative)
plt.xlabel("Words per speaker turn")
plt.ylabel("Cumulative fraction of turns")
plt.title("Cumulative Distribution of Turn Lengths")
plt.grid(True)
plt.show()


# Across patients

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams['font.family'] = 'Arial'
from pathlib import Path

FIGURES_DIR = Path("/Users/aniluchavez/Documents/Language/Figures/Figure1")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)  # ensure it exists

OUTPUT_PATH = FIGURES_DIR / "all_patients_turn_histogram.eps"

BASE_DIR = Path("/Users/aniluchavez/Documents/Language/Python/Final_words_english_only_BERT")

all_turn_summaries = []

for patient_folder in BASE_DIR.glob("*_words_english_only"):
    print(f"\n📌 Processing folder: {patient_folder.name}")

    # Try to find the Excel file
    excel_files = [f for f in patient_folder.glob("*filtered_used_rows_withNP.xlsx") if not f.name.startswith("~$")]
    if not excel_files:
        print("⚠️  No matching Excel found, skipping!")
        continue

    excel_path = excel_files[0]
    print(f"✅ Found Excel: {excel_path.name}")

    # Load Excel
    df = pd.read_excel(excel_path)

    # Auto-detect speaker columns
    speaker_columns = [col for col in df.columns if col.startswith("Speaker")]
    if not speaker_columns:
        print("⚠️  No speaker columns found, skipping!")
        continue

# Dynamically choose id_vars based on what is actually present
    potential_id_vars = ["onset", "offset", "Duration", "regress_dur"]
    id_vars = [col for col in potential_id_vars if col in df.columns]

    # Melt wide to long
    long_df = df.melt(
        id_vars=id_vars,
        value_vars=speaker_columns,
        var_name="speaker",
        value_name="word"
    )
    long_df = long_df.dropna(subset=["word"]).reset_index(drop=True)


    if long_df.empty:
        print("⚠️  No words after melting, skipping!")
        continue

    # Sort by onset
    long_df = long_df.sort_values(by="onset").reset_index(drop=True)

    # Detect speaker changes
    long_df["speaker_shift"] = long_df["speaker"] != long_df["speaker"].shift()
    long_df["turn_id"] = long_df["speaker_shift"].cumsum()

    # Summarize turns
    turn_summary = long_df.groupby("turn_id").agg({
        "speaker": "first",
        "onset": "min",
        "word": "count"
    }).rename(columns={"word": "n_words"}).reset_index()

    # Add patient ID column
    patient_id = patient_folder.name.replace("_words_english_only", "")
    turn_summary["patient"] = patient_id

    # Store
    all_turn_summaries.append(turn_summary)

# Concatenate all patients
if all_turn_summaries:
    pooled_df = pd.concat(all_turn_summaries, ignore_index=True)
    print("\n✅ Successfully pooled all patients!")
    print(pooled_df.head())

    # Save to CSV
    pooled_df.to_csv("all_patients_turn_summary.csv", index=False)
    print("\n✅ Full pooled report saved to 'all_patients_turn_summary.csv'")

    # Print overall stats
    print("\n=== Overall Words per Turn Stats ===")
    print(pooled_df["n_words"].describe())

    plt.rcParams.update({'font.size': 30})

    fig, ax = plt.subplots(figsize=(10, 8))
    plt.hist(pooled_df["n_words"], bins=100, edgecolor='black', color='#CC5500')
    plt.xlabel("Words per speaker turn")
    plt.ylabel("Number of turns")
    # plt.title("Histogram of Words per Speaker Turn (All Patients)")
    plt.xlim(0, 80)  # Focus on the dense region

    # Style tweaks: remove top and right spines
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

 

    plt.tight_layout()
    
    
    plt.savefig(OUTPUT_PATH, format='eps')
    print(f"\n✅ Histogram saved to: {OUTPUT_PATH}")

    plt.show()


else:
    print("\n❌ No valid data found in any patient folders.")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams['font.family'] = 'Arial'

# === Paths ===
FIGURES_DIR = Path("/Users/aniluchavez/Documents/Language/Figures/Figure1")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH_SELF = FIGURES_DIR / "turn_histogram_speaker1_neutral1.eps"
OUTPUT_PATH_OTHER = FIGURES_DIR / "turn_histogram_other_speakers_neutral2.eps"
BASE_DIR = Path("/Users/aniluchavez/Documents/Language/Python/Final_words_english_only_BERT")

# === Storage ===
speaker1_turns = []
other_speaker_turns = []

for patient_folder in BASE_DIR.glob("*_words_english_only"):
    print(f"\n📌 Processing folder: {patient_folder.name}")

    excel_files = [f for f in patient_folder.glob("*filtered_used_rows_withNP.xlsx") if not f.name.startswith("~$")]
    if not excel_files:
        print("⚠️  No matching Excel found, skipping!")
        continue

    df = pd.read_excel(excel_files[0])
    speaker_columns = [col for col in df.columns if col.startswith("Speaker")]
    if not speaker_columns:
        print("⚠️  No speaker columns found, skipping!")
        continue

    potential_id_vars = ["onset", "offset", "Duration", "regress_dur"]
    id_vars = [col for col in potential_id_vars if col in df.columns]

    long_df = df.melt(
        id_vars=id_vars,
        value_vars=speaker_columns,
        var_name="speaker",
        value_name="word"
    )
    long_df = long_df.dropna(subset=["word"]).sort_values(by="onset").reset_index(drop=True)

    if long_df.empty:
        print("⚠️  No words after melting, skipping!")
        continue

    long_df["speaker_shift"] = long_df["speaker"] != long_df["speaker"].shift()
    long_df["turn_id"] = long_df["speaker_shift"].cumsum()

    turn_summary = long_df.groupby("turn_id").agg({
        "speaker": "first",
        "onset": "min",
        "word": "count"
    }).rename(columns={"word": "n_words"}).reset_index()

    patient_id = patient_folder.name.replace("_words_english_only", "")
    turn_summary["patient"] = patient_id

    speaker1_turns.append(turn_summary[turn_summary["speaker"] == "Speaker1"])
    other_speaker_turns.append(turn_summary[turn_summary["speaker"] != "Speaker1"])

# === Combine
df_s1 = pd.concat(speaker1_turns, ignore_index=True) if speaker1_turns else pd.DataFrame()
df_others = pd.concat(other_speaker_turns, ignore_index=True) if other_speaker_turns else pd.DataFrame()

# === Plot Speaker1 ===
# if not df_s1.empty:
#     plt.rcParams.update({'font.size': 26})
#     fig, ax = plt.subplots(figsize=(10, 8))

#     plt.hist(df_s1["n_words"], bins=range(0, 81, 2), color="#D07A18", edgecolor='black')
#     plt.xlabel("Words per speaker turn (Speaker1)")
#     plt.ylabel("Number of turns")
#     plt.xlim(0, 80)
#     ax.spines['top'].set_visible(False)
#     ax.spines['right'].set_visible(False)

#     plt.tight_layout()
#     plt.savefig(OUTPUT_PATH_SELF, format='eps')
#     print(f"✅ Speaker1 histogram saved to: {OUTPUT_PATH_SELF}")
#     plt.show()

# # === Plot Other Speakers ===
# if not df_others.empty:
#     plt.rcParams.update({'font.size': 26})
#     fig, ax = plt.subplots(figsize=(10, 8))

#     plt.hist(df_others["n_words"], bins=range(0, 81, 2), color="#E04926", edgecolor='black')
#     plt.xlabel("Words per speaker turn (Other Speakers)")
#     plt.ylabel("Number of turns")
#     plt.xlim(0, 80)
#     ax.spines['top'].set_visible(False)
#     ax.spines['right'].set_visible(False)

#     plt.tight_layout()
#     plt.savefig(OUTPUT_PATH_OTHER, format='eps')
#     print(f"✅ Other speaker histogram saved to: {OUTPUT_PATH_OTHER}")
#     plt.show()

def summarize_turns(df, label, outdir):
    """
    Print and save summary stats for turns dataframe (expects columns 'n_words' and 'patient').
    Returns (summary_dict, per_patient_df).
    """
    print(f"\n--- SUMMARY: {label} ---")
    if df.empty:
        print("⚠️  No data found for", label)
        return None, None

    # overall stats
    mean = df["n_words"].mean()
    median = df["n_words"].median()
    std = df["n_words"].std()
    n_turns = int(len(df))
    n_patients = df["patient"].nunique()
    q25 = df["n_words"].quantile(0.25)
    q75 = df["n_words"].quantile(0.75)
    mn = int(df["n_words"].min())
    mx = int(df["n_words"].max())

    print(f"Total turns: {n_turns:,}  (from {n_patients} patients)")
    print(f"Mean words / turn : {mean:.2f}")
    print(f"Median words / turn : {median:.2f}")
    print(f"Std dev : {std:.2f}")
    print(f"25th pct / 75th pct : {q25:.1f} / {q75:.1f}")
    print(f"Min / Max words in a turn : {mn} / {mx}")

    # per-patient summary
    per_patient = (
        df.groupby("patient")["n_words"]
        .agg(['count','mean','median','std'])
        .rename(columns={'count':'n_turns','mean':'mean_n_words','median':'median_n_words','std':'std_n_words'})
        .reset_index()
    )
    # round for readability
    per_patient['mean_n_words'] = per_patient['mean_n_words'].round(2)
    per_patient['median_n_words'] = per_patient['median_n_words'].round(2)
    per_patient['std_n_words'] = per_patient['std_n_words'].round(2)

    # save per-patient CSV and overall summary JSON-like CSV
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    per_patient_f = outdir / f"{label.replace(' ','_')}_turns_by_patient.csv"
    per_patient.to_csv(per_patient_f, index=False)
    print(f"Saved per-patient summary to: {per_patient_f}")

    # save an overall small summary csv
    overall_summary = {
        'label': label,
        'total_turns': n_turns,
        'n_patients': n_patients,
        'mean_n_words': round(mean, 4),
        'median_n_words': round(median, 4),
        'std_n_words': round(std, 4),
        'q25': float(q25),
        'q75': float(q75),
        'min_n_words': mn,
        'max_n_words': mx
    }
    overall_f = outdir / f"{label.replace(' ','_')}_summary.csv"
    pd.DataFrame([overall_summary]).to_csv(overall_f, index=False)
    print(f"Saved overall summary to: {overall_f}")

    # print a few extremal examples (helpful sanity check)
    print("\nTop 5 longest turns (words) — patient, n_words, onset (if available):")
    try:
        print(df.sort_values("n_words", ascending=False).head(5)[["patient","n_words","onset"]].to_string(index=False))
    except Exception:
        print(df.sort_values("n_words", ascending=False).head(5)[["patient","n_words"]].to_string(index=False))

    print("\nTop 5 shortest turns (words):")
    print(df.sort_values("n_words", ascending=True).head(5)[["patient","n_words"]].to_string(index=False))

    return overall_summary, per_patient

# === Summarize and print ===
summary_s1, per_patient_s1 = summarize_turns(df_s1, "Speaker1", FIGURES_DIR)
summary_others, per_patient_others = summarize_turns(df_others, "Other_Speakers", FIGURES_DIR)

# === (existing plotting code follows) ===
# === Plot Speaker1 ===
if not df_s1.empty:
    plt.rcParams.update({'font.size': 26})
    fig, ax = plt.subplots(figsize=(10, 8))

    plt.hist(df_s1["n_words"], bins=range(0, 81, 2), color="#D07A18", edgecolor='black')
    plt.xlabel("Words per speaker turn (Speaker1)")
    plt.ylabel("Number of turns")
    plt.xlim(0, 80)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig(OUTPUT_PATH_SELF, format='eps')
    print(f"✅ Speaker1 histogram saved to: {OUTPUT_PATH_SELF}")
    plt.show()

# === Plot Other Speakers ===
if not df_others.empty:
    plt.rcParams.update({'font.size': 26})
    fig, ax = plt.subplots(figsize=(10, 8))

    plt.hist(df_others["n_words"], bins=range(0, 81, 2), color="#E04926", edgecolor='black')
    plt.xlabel("Words per speaker turn (Other Speakers)")
    plt.ylabel("Number of turns")
    plt.xlim(0, 80)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig(OUTPUT_PATH_OTHER, format='eps')
    print(f"✅ Other speaker histogram saved to: {OUTPUT_PATH_OTHER}")
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

OUTPUT_PATH = FIGURES_DIR / "all_patients_cumulativedensity.eps"

# Use all pooled turn lengths
all_sizes = pooled_df["n_words"].values

# Compute quantiles
q25 = np.percentile(all_sizes, 25)
q50 = np.percentile(all_sizes, 50)
q75 = np.percentile(all_sizes, 75)

print(f"25th percentile: {q25:.2f} words")
print(f"50th percentile (median): {q50:.2f} words")
print(f"75th percentile: {q75:.2f} words")

# Sort them for cumulative
sorted_sizes = np.sort(all_sizes)
cumulative = np.arange(1, len(sorted_sizes) + 1) / len(sorted_sizes)

# Set global font size for labels/ticks
plt.rcParams.update({'font.size': 30})

# Create figure and axis
fig, ax = plt.subplots(figsize=(10, 8))
ax.plot(sorted_sizes, cumulative, color='#CC5500', label='Cumulative Distribution')

# Labels and axes
ax.set_xlabel("Words per speaker turn")
ax.set_ylabel("Cumulative fraction of turns")

# Zoom in x-axis
ax.set_xlim(0, 125)

# Add quantile lines with *smaller* text
# Add quantile lines with staggered y-positions
# Add only 50% and 75% quantile lines with staggered y-positions
quantiles = [q50, q75]
labels = ['50%', '75%']
ys = [0.12, 0.05]  # vertical positions for text labels

for q, label, y in zip(quantiles, labels, ys):
    ax.axvline(x=q, color='gray', linestyle='--', alpha=0.7)
    ax.text(q + 2, y, f'{label}: {q:.0f}', rotation=90, va='bottom',
            color='gray', fontsize=16)



# Clean style: remove top/right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Remove grid lines
ax.grid(False)

# Smaller legend font
ax.legend(fontsize=18)

plt.tight_layout()


plt.savefig(OUTPUT_PATH, format='eps')



print(f"\n✅ Histogram saved to: {OUTPUT_PATH}")
plt.show()